---
## ① Install Dependencies
This takes ~60 seconds on first run.

In [ ]:
%%capture
!pip install skyfield sgp4 xgboost imbalanced-learn plotly tqdm fastapi uvicorn pydantic
print('✅ All packages installed')

In [ ]:
# Verify key imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestClassifier
from skyfield.api import EarthSatellite, load
from sgp4.api import Satrec
print('✅ All imports successful')

---
## ② Download TLE Satellite Data from Celestrak

In [ ]:
import requests, time, logging, os, json
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(levelname)s  %(message)s')
log = logging.getLogger(__name__)

os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('models', exist_ok=True)

EARTH_RADIUS_KM = 6371.0
MU              = 398600.4418

# ✅ FIXED: New Celestrak GP query format
TLE_SOURCES = {
    'active':   'https://celestrak.org/NORAD/elements/gp.php?GROUP=active&FORMAT=tle',
    'stations': 'https://celestrak.org/NORAD/elements/gp.php?GROUP=stations&FORMAT=tle',
    'starlink': 'https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle',
}

def download_tle(url, save_path, timeout=30):
    headers = {'User-Agent': 'space-debris-ml-colab/1.0 (academic research)'}
    try:
        resp = requests.get(url, headers=headers, timeout=timeout)
        resp.raise_for_status()
        if len(resp.text.strip()) < 100:
            print(f'  ⚠️  Empty response from {url}')
            return False
        Path(save_path).write_text(resp.text, encoding='utf-8')
        print(f'  ✅ Downloaded {len(resp.text):,} bytes → {save_path}')
        return True
    except Exception as e:
        print(f'  ⚠️  Failed: {e}')
        return False

def estimate_altitude(mean_motion_rpm):
    if mean_motion_rpm <= 0:
        return 0.0
    n = mean_motion_rpm / 60.0
    r = (MU / (n**2))**(1/3)
    return max(r - EARTH_RADIUS_KM, 0.0)

def parse_tle_file(file_path):
    from sgp4.api import Satrec
    lines = Path(file_path).read_text(encoding='utf-8').splitlines()
    lines = [l.strip() for l in lines if l.strip()]
    records = []
    i = 0
    while i + 2 < len(lines):
        name, line1, line2 = lines[i], lines[i+1], lines[i+2]
        if not (line1.startswith('1 ') and line2.startswith('2 ')):
            i += 1
            continue
        try:
            sat = Satrec.twoline2rv(line1, line2)
            pi  = 3.141592653589793
            records.append({
                'name':            name,
                'catalog_number':  sat.satnum,
                'tle_line1':       line1,
                'tle_line2':       line2,
                'inclination_deg': sat.inclo  * (180/pi),
                'raan_deg':        sat.nodeo  * (180/pi),
                'eccentricity':    sat.ecco,
                'arg_perigee_deg': sat.argpo  * (180/pi),
                'mean_motion_rpm': sat.no_kozai,
                'bstar_drag':      sat.bstar,
            })
        except:
            pass
        i += 3
    return pd.DataFrame(records)

# ── Download & parse ───────────────────────────────────────────────────────────
print('📡 Downloading TLE data from Celestrak ...')
frames = []
for name, url in TLE_SOURCES.items():
    path = f'data/raw/{name}.tle'
    if download_tle(url, path):
        df = parse_tle_file(path)
        if len(df) > 0:
            frames.append(df)
            print(f'     → Parsed {len(df)} satellites from {name}')
    time.sleep(1.5)   # be polite to Celestrak

if not frames:
    raise RuntimeError("❌ All downloads failed. Check your internet connection in Colab.")

sat_df = pd.concat(frames, ignore_index=True).drop_duplicates('catalog_number')
sat_df['altitude_km'] = sat_df['mean_motion_rpm'].apply(estimate_altitude)
sat_df = sat_df[sat_df['altitude_km'] < 2000].reset_index(drop=True)

if len(sat_df) > 300:
    sat_df = sat_df.sample(300, random_state=42).reset_index(drop=True)

sat_df.to_csv('data/raw/satellites.csv', index=False)
print(f'\n✅ {len(sat_df)} LEO satellites saved to data/raw/satellites.csv')
sat_df[['name','inclination_deg','altitude_km','eccentricity']].head(8)

---
## ③ Orbital Element Distribution — EDA

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='darkgrid', palette='muted')

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('LEO Satellite Orbital Element Distributions', fontsize=15, fontweight='bold')

plots = [
    ('inclination_deg', 'Inclination (°)', 'steelblue'),
    ('altitude_km',     'Altitude (km)',   'seagreen'),
    ('eccentricity',    'Eccentricity',    'coral'),
    ('raan_deg',        'RAAN (°)',         'mediumpurple'),
    ('arg_perigee_deg', 'Arg. Perigee (°)','goldenrod'),
    ('bstar_drag',      'BSTAR Drag',      'tomato'),
]
for ax, (col, title, color) in zip(axes.ravel(), plots):
    sat_df[col].dropna().hist(ax=ax, bins=30, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.savefig('models/eda_distributions.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ EDA plot saved')

---
## ④ Orbit Propagation (SGP4 via Skyfield)

In [ ]:
from datetime import datetime, timezone
from skyfield.api import EarthSatellite, load, wgs84
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

TS = load.timescale()

def propagate_satellite(name, line1, line2, duration_min=90, step_min=5):
    sat     = EarthSatellite(line1, line2, name, TS)
    t_now   = datetime.now(timezone.utc)
    t0      = TS.from_datetime(t_now)
    n_steps = int(duration_min / step_min) + 1
    minutes = np.linspace(0, duration_min, n_steps)
    times   = TS.tt_jd(t0.tt + minutes / 1440.0)

    geo = sat.at(times)
    pos = geo.position.km          # shape (3, n_steps)
    vel = geo.velocity.km_per_s    # shape (3, n_steps)

    # ✅ FIX: use geographic() instead of subpoint_of() for lat/lon/alt
    #    subpoint_of() can crash on re-entered / decayed satellites
    rows = []
    for i in range(n_steps):
        x, y, z = float(pos[0,i]), float(pos[1,i]), float(pos[2,i])

        # Compute altitude from ECI radius (no wgs84 call needed)
        r_km  = (x**2 + y**2 + z**2) ** 0.5
        alt   = r_km - 6371.0

        # Skip if satellite has decayed (underground) or NaN
        if alt < 0 or alt != alt:
            return None

        rows.append({
            'satellite_name': name,
            'time_min':  float(minutes[i]),
            'x_km':      x,
            'y_km':      y,
            'z_km':      z,
            'vx_kms':    float(vel[0,i]),
            'vy_kms':    float(vel[1,i]),
            'vz_kms':    float(vel[2,i]),
            'alt_km':    alt,
        })
    return pd.DataFrame(rows)

# ── Propagate ──────────────────────────────────────────────────────────────────
MAX_SATS = 60
subset   = sat_df.head(MAX_SATS)
frames   = []
skipped  = 0

print(f'🔭 Propagating {MAX_SATS} satellites ...')
for _, row in tqdm(subset.iterrows(), total=MAX_SATS):
    try:
        traj = propagate_satellite(
            row['name'], row['tle_line1'], row['tle_line2']
        )
        if traj is not None and len(traj) > 0:
            frames.append(traj)
        else:
            skipped += 1
    except Exception as e:
        skipped += 1

print(f'\n✅ Success: {len(frames)} satellites  |  Skipped (decayed/invalid): {skipped}')

if len(frames) == 0:
    raise RuntimeError(
        "❌ No satellites propagated. Your TLE data may be stale.\n"
        "Try re-running Cell ② to re-download fresh TLEs."
    )

traj_df = pd.concat(frames, ignore_index=True)
traj_df.to_csv('data/processed/trajectories.csv', index=False)
print(f'   Trajectory rows: {len(traj_df):,}')
print(f'   Altitude range:  {traj_df["alt_km"].min():.0f} – {traj_df["alt_km"].max():.0f} km')
traj_df.head()

---
## ⑤ Feature Engineering — Pairwise Collision Features

In [ ]:
import itertools

def angle_between(v1, v2):
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 == 0 or n2 == 0: return 0.0
    return float(np.degrees(np.arccos(np.clip(np.dot(v1,v2)/(n1*n2), -1, 1))))

# Index trajectories
grouped   = {n: g.reset_index(drop=True) for n, g in traj_df.groupby('satellite_name')}
sat_names = list(grouped.keys())

# ✅ FIX: deduplicate by name BEFORE setting index, then convert to plain dict
#    so lookups always return a scalar, never a Series or DataFrame
sat_meta = (
    sat_df[['name','inclination_deg','raan_deg','eccentricity']]
    .drop_duplicates('name')
    .set_index('name')
    .to_dict('index')   # {'SAT_NAME': {'inclination_deg': 51.6, ...}, ...}
)

traj_df['speed'] = np.linalg.norm(traj_df[['vx_kms','vy_kms','vz_kms']].values, axis=1)
spd_lookup = traj_df.groupby('satellite_name')['speed'].mean().to_dict()
alt_lookup = traj_df.groupby('satellite_name')['alt_km'].mean().to_dict()

pairs = list(itertools.combinations(sat_names, 2))
MAX_PAIRS = 3000
if len(pairs) > MAX_PAIRS:
    rng   = np.random.default_rng(42)
    pairs = [pairs[i] for i in rng.choice(len(pairs), MAX_PAIRS, replace=False)]

print(f'⚙️  Computing features for {len(pairs):,} satellite pairs ...')

records = []
for sat_a, sat_b in tqdm(pairs):
    ta, tb = grouped[sat_a], grouped[sat_b]
    n = min(len(ta), len(tb))
    ta, tb = ta.iloc[:n], tb.iloc[:n]

    pa = ta[['x_km','y_km','z_km']].values
    pb = tb[['x_km','y_km','z_km']].values
    va = ta[['vx_kms','vy_kms','vz_kms']].values
    vb = tb[['vx_kms','vy_kms','vz_kms']].values

    dists  = np.linalg.norm(pa - pb, axis=1)
    speeds = np.linalg.norm(va - vb, axis=1)
    idx    = int(np.argmin(dists))

    # ✅ FIX: plain dict .get() always returns a scalar float
    ma = sat_meta.get(sat_a, {})
    mb = sat_meta.get(sat_b, {})

    records.append({
        'sat_a':                 sat_a,
        'sat_b':                 sat_b,
        'min_distance_km':       float(dists[idx]),
        'mean_distance_km':      float(dists.mean()),
        'dist_std_km':           float(dists.std()),
        'rel_velocity_kms':      float(speeds[idx]),
        'mean_rel_velocity_kms': float(speeds.mean()),
        'incl_diff_deg':         abs(float(ma.get('inclination_deg', 0)) - float(mb.get('inclination_deg', 0))),
        'raan_diff_deg':         abs(float(ma.get('raan_deg', 0))        - float(mb.get('raan_deg', 0))),
        'ecc_diff':              abs(float(ma.get('eccentricity', 0))    - float(mb.get('eccentricity', 0))),
        'alt_diff_km':           abs(alt_lookup.get(sat_a, 0)            - alt_lookup.get(sat_b, 0)),
        'combined_altitude_km':  (alt_lookup.get(sat_a, 0) + alt_lookup.get(sat_b, 0)) / 2,
        'mean_speed_kms':        (spd_lookup.get(sat_a, 0) + spd_lookup.get(sat_b, 0)) / 2,
        'time_to_tca_min':       float(ta['time_min'].iloc[idx]),
        'approach_angle_deg':    angle_between(va[idx], vb[idx]),
    })

feat_df = pd.DataFrame(records)
feat_df.to_csv('data/processed/pair_features.csv', index=False)
print(f'\n✅ Feature matrix: {feat_df.shape}')
feat_df.describe().round(2)

---
## ⑥ Dataset Labelling, Balancing & ML Training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics         import roc_auc_score, classification_report, confusion_matrix
from xgboost                 import XGBClassifier
from imblearn.over_sampling  import SMOTE
import joblib, json

# ── Label by distance threshold ────────────────────────────────────────────────
THRESHOLD_KM = 5.0
feat_df['collision_risk'] = (feat_df['min_distance_km'] < THRESHOLD_KM).astype(int)
feat_df['risk_category']  = 'Low Risk'
feat_df.loc[feat_df['min_distance_km'] < 20, 'risk_category'] = 'Medium Risk'
feat_df.loc[feat_df['min_distance_km'] < THRESHOLD_KM, 'risk_category'] = 'High Risk'

print('📊 Class distribution before balancing:')
print(feat_df['risk_category'].value_counts().to_string())

# ── Inject synthetic high-risk pairs ──────────────────────────────────────────
rng = np.random.default_rng(42)
n_synth = 300
synth = feat_df.sample(n=n_synth, replace=True, random_state=42).copy()
synth['min_distance_km']      = rng.uniform(0.2, 4.9, n_synth)
synth['mean_distance_km']     = synth['min_distance_km'] + rng.uniform(1,10,n_synth)
synth['rel_velocity_kms']     = rng.uniform(6.0, 14.0, n_synth)
synth['approach_angle_deg']   = rng.uniform(150, 180, n_synth)
synth['alt_diff_km']          = rng.uniform(0, 5, n_synth)
synth['collision_risk']       = 1
synth['risk_category']        = 'High Risk'
synth['sat_a'] = 'SYNTH_A'
synth['sat_b'] = 'SYNTH_B'

labelled = pd.concat([feat_df, synth], ignore_index=True)
labelled.to_csv('data/processed/labelled_dataset.csv', index=False)

print(f'\n📊 After synthetic injection:')
print(labelled['risk_category'].value_counts().to_string())

In [ ]:
# ── Train / test split ─────────────────────────────────────────────────────────
FEATURE_COLS = [
    'min_distance_km','mean_distance_km','dist_std_km',
    'rel_velocity_kms','mean_rel_velocity_kms',
    'incl_diff_deg','raan_diff_deg','ecc_diff',
    'alt_diff_km','combined_altitude_km','mean_speed_kms',
    'time_to_tca_min','approach_angle_deg'
]

clean = labelled.dropna(subset=FEATURE_COLS + ['collision_risk'])
X = clean[FEATURE_COLS].values
y = clean['collision_risk'].values

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                            stratify=y, random_state=42)
scaler  = StandardScaler()
X_tr    = scaler.fit_transform(X_tr)
X_te    = scaler.transform(X_te)

# SMOTE balancing
sm = SMOTE(random_state=42)
X_tr, y_tr = sm.fit_resample(X_tr, y_tr)

print(f'✅ Train: {X_tr.shape}  |  Test: {X_te.shape}')
print(f'   Positive rate — train: {y_tr.mean():.3f}  test: {y_te.mean():.3f}')

In [ ]:
# ── Train all models ───────────────────────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, random_state=42),
    'XGBoost':             XGBClassifier(n_estimators=300, learning_rate=0.05, scale_pos_weight=10,
                                          eval_metric='logloss', random_state=42, n_jobs=-1),
}

results = {}
print('🤖 Training models ...\n')
print(f'{"Model":<25} {"ROC-AUC":>8} {"Precision":>10} {"Recall":>8} {"F1":>8}')
print('─' * 65)

for name, clf in models.items():
    clf.fit(X_tr, y_tr)
    y_prob = clf.predict_proba(X_te)[:,1]
    y_pred = clf.predict(X_te)
    auc    = roc_auc_score(y_te, y_prob)
    rep    = classification_report(y_te, y_pred, output_dict=True, zero_division=0)
    results[name] = {'model': clf, 'auc': auc, 'proba': y_prob, 'pred': y_pred,
                     'p': rep['1']['precision'], 'r': rep['1']['recall'], 'f1': rep['1']['f1-score']}
    print(f'{name:<25} {auc:>8.4f} {rep["1"]["precision"]:>10.3f} {rep["1"]["recall"]:>8.3f} {rep["1"]["f1-score"]:>8.3f}')

best_name = max(results, key=lambda k: results[k]['auc'])
print(f'\n🏆 Best model: {best_name}  (AUC = {results[best_name]["auc"]:.4f})')

In [ ]:
# ── Save best model ────────────────────────────────────────────────────────────
joblib.dump(results[best_name]['model'], 'models/best_model.joblib')
joblib.dump(scaler, 'models/scaler.joblib')

meta = {'best_model': best_name, 'roc_auc': results[best_name]['auc'],
        'precision': results[best_name]['p'], 'recall': results[best_name]['r'],
        'f1': results[best_name]['f1'], 'feature_names': FEATURE_COLS}
with open('models/model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('✅ Model saved to models/')

---
## ⑦ Evaluation Plots

In [ ]:
from sklearn.metrics import RocCurveDisplay
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC curves
for name, res in results.items():
    RocCurveDisplay.from_predictions(y_te, res['proba'],
        name=f"{name} ({res['auc']:.3f})", ax=axes[0])
axes[0].set_title('ROC Curves — All Models', fontweight='bold')

# Confusion matrix — best model
cm = confusion_matrix(y_te, results[best_name]['pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Risk','Risk'], yticklabels=['No Risk','Risk'],
            ax=axes[1])
axes[1].set_title(f'Confusion Matrix — {best_name}', fontweight='bold')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')

# Feature importance
clf = results[best_name]['model']
if hasattr(clf, 'feature_importances_'):
    fi = pd.Series(clf.feature_importances_, index=FEATURE_COLS).sort_values()
    fi.plot(kind='barh', ax=axes[2], color='steelblue')
    axes[2].set_title(f'Feature Importance — {best_name}', fontweight='bold')

plt.suptitle('Model Evaluation Dashboard', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('models/evaluation_plots.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Evaluation plots saved')

---
## ⑧ Risk Prediction & Monte Carlo Simulation

In [ ]:
# ── Risk predictor class ───────────────────────────────────────────────────────
class CollisionRiskPredictor:
    def __init__(self):
        self.model         = joblib.load('models/best_model.joblib')
        self.scaler        = joblib.load('models/scaler.joblib')
        self.feature_names = FEATURE_COLS

    def _prepare(self, features):
        if isinstance(features, dict):
            arr = np.array([features.get(f, 0.0) for f in self.feature_names], dtype=float)
        else:
            arr = np.asarray(features, dtype=float).ravel()
        return self.scaler.transform(arr.reshape(1, -1))

    def predict(self, features):
        p   = float(self.model.predict_proba(self._prepare(features))[0, 1])
        cat = 'High Risk' if p >= 0.5 else ('Medium Risk' if p >= 0.2 else 'Low Risk')
        return {'collision_probability': round(p, 4),
                'risk_category': cat, 'risk_score': int(p * 100)}

    def monte_carlo(self, features, n_sims=1000, noise_frac=0.05):
        x_base = self.scaler.inverse_transform(self._prepare(features))
        rng    = np.random.default_rng(42)
        probs  = np.array([
            float(self.model.predict_proba(
                self.scaler.transform(x_base + rng.normal(0, noise_frac * np.abs(x_base)))
            )[0, 1])
            for _ in range(n_sims)
        ])
        mean_p = float(probs.mean())
        return {
            'mean_probability':   round(mean_p, 4),
            'std_probability':    round(float(probs.std()), 4),
            'p5':                 round(float(np.percentile(probs, 5)),  4),
            'p95':                round(float(np.percentile(probs, 95)), 4),
            'risk_category':      'High Risk' if mean_p >= 0.5 else ('Medium Risk' if mean_p >= 0.2 else 'Low Risk'),
            'high_risk_fraction': round(float((probs > 0.5).mean()), 4),
            'n_simulations':      n_sims,
            '_probs':             probs,
        }

predictor = CollisionRiskPredictor()
print('✅ Predictor loaded')

# ── Test: High-risk scenario ───────────────────────────────────────────────────
high_risk = {
    'min_distance_km': 2.5, 'mean_distance_km': 8.0, 'dist_std_km': 3.5,
    'rel_velocity_kms': 12.1, 'mean_rel_velocity_kms': 9.5, 'incl_diff_deg': 5.2,
    'raan_diff_deg': 12.0, 'ecc_diff': 0.01, 'alt_diff_km': 1.8,
    'combined_altitude_km': 580.0, 'mean_speed_kms': 7.6,
    'time_to_tca_min': 32.0, 'approach_angle_deg': 168.0,
}

low_risk = {
    'min_distance_km': 450.0, 'mean_distance_km': 800.0, 'dist_std_km': 120.0,
    'rel_velocity_kms': 2.1, 'mean_rel_velocity_kms': 1.8, 'incl_diff_deg': 45.0,
    'raan_diff_deg': 90.0, 'ecc_diff': 0.05, 'alt_diff_km': 200.0,
    'combined_altitude_km': 700.0, 'mean_speed_kms': 7.4,
    'time_to_tca_min': 78.0, 'approach_angle_deg': 25.0,
}

print('\n── Point Estimates ─────────────────────────────────────────────')
for label, scenario in [('HIGH RISK scenario', high_risk), ('LOW RISK scenario', low_risk)]:
    r = predictor.predict(scenario)
    emoji = '🔴' if r['risk_category'] == 'High Risk' else ('🟡' if r['risk_category'] == 'Medium Risk' else '🟢')
    print(f'  {emoji} {label}: P={r["collision_probability"]:.4f}  [{r["risk_category"]}]  Score={r["risk_score"]}/100')

In [ ]:
# ── Monte Carlo simulation ─────────────────────────────────────────────────────
print('🎲 Running Monte Carlo simulation (1000 iterations) ...')
mc = predictor.monte_carlo(high_risk, n_sims=1000)

print(f'\n── Monte Carlo Results ─────────────────────────────────────────')
for k, v in mc.items():
    if k != '_probs':
        print(f'  {k:<25}: {v}')

# Plot distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

probs = mc['_probs']
axes[0].hist(probs, bins=50, color='#e63946', edgecolor='white', alpha=0.85)
axes[0].axvline(mc['mean_probability'], color='white',  lw=2, label=f"Mean={mc['mean_probability']:.3f}")
axes[0].axvline(mc['p5'],  color='yellow', lw=1.5, ls='--', label=f"P5={mc['p5']:.3f}")
axes[0].axvline(mc['p95'], color='yellow', lw=1.5, ls='--', label=f"P95={mc['p95']:.3f}")
axes[0].set_facecolor('#0d1117'); axes[0].figure.set_facecolor('#0d1117')
axes[0].tick_params(colors='white'); axes[0].xaxis.label.set_color('white'); axes[0].yaxis.label.set_color('white')
axes[0].set_title('Monte Carlo — Collision Probability Distribution', color='white', fontweight='bold')
axes[0].set_xlabel('Collision Probability', color='white')
axes[0].legend(facecolor='#1a1a2e', labelcolor='white')

# Risk score comparison bar chart
scenarios = ['High Risk\nScenario', 'Low Risk\nScenario']
scores    = [predictor.predict(high_risk)['risk_score'],
             predictor.predict(low_risk)['risk_score']]
colors    = ['#e63946', '#2dc653']
axes[1].bar(scenarios, scores, color=colors, edgecolor='white', width=0.5)
axes[1].set_ylim(0, 100)
axes[1].set_ylabel('Risk Score (0–100)', color='white')
axes[1].set_title('Risk Score Comparison', color='white', fontweight='bold')
axes[1].set_facecolor('#0d1117')
axes[1].tick_params(colors='white')
for bar, score in zip(axes[1].patches, scores):
    axes[1].text(bar.get_x() + bar.get_width()/2, score + 2, f'{score}',
                ha='center', color='white', fontweight='bold', fontsize=14)

fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.savefig('models/monte_carlo_results.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

---
## ⑨ Interactive 3-D Orbital Visualisation

In [ ]:
import colorsys, plotly.graph_objects as go

def earth_sphere(n=50):
    R = 6371.0
    u = np.linspace(0, 2*np.pi, n)
    v = np.linspace(0, np.pi, n)
    return go.Surface(
        x=R*np.outer(np.cos(u), np.sin(v)),
        y=R*np.outer(np.sin(u), np.sin(v)),
        z=R*np.outer(np.ones(n), np.cos(v)),
        colorscale=[[0,'#1a4db5'],[1,'#4d9de0']],
        showscale=False, opacity=0.65, name='Earth', hoverinfo='skip'
    )

def palette(n):
    colors = []
    for i in range(n):
        r,g,b = colorsys.hsv_to_rgb(i/max(n,1), 0.7, 0.95)
        colors.append(f'#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}')
    return colors

fig = go.Figure()
fig.add_trace(earth_sphere())

sat_names = traj_df['satellite_name'].unique()[:25]
pal = palette(len(sat_names))
for i, name in enumerate(sat_names):
    t = traj_df[traj_df['satellite_name'] == name]
    fig.add_trace(go.Scatter3d(
        x=t['x_km'], y=t['y_km'], z=t['z_km'],
        mode='lines', line=dict(color=pal[i], width=2),
        name=name, hoverinfo='name', opacity=0.8
    ))

# Add risk markers for sampled pairs
risk_sample = labelled[labelled['risk_category'] == 'High Risk'].head(15)
for _, row in risk_sample.iterrows():
    # Approximate TCA position via one of the satellite's trajectory midpoints
    t = traj_df[traj_df['satellite_name'] == row['sat_a']]
    if len(t) == 0: continue
    mid = t.iloc[len(t)//2]
    fig.add_trace(go.Scatter3d(
        x=[mid['x_km']], y=[mid['y_km']], z=[mid['z_km']],
        mode='markers',
        marker=dict(size=7, color='#ff2200', symbol='diamond',
                    line=dict(color='white', width=1)),
        name='High Risk TCA',
        hovertext=f"{row['sat_a']} ↔ {row['sat_b']}<br>d={row['min_distance_km']:.1f} km",
        hoverinfo='text', showlegend=False
    ))

fig.update_layout(
    title=dict(text='🛰️  Space Debris Collision Risk — LEO Orbital View',
               font=dict(size=18, color='white')),
    paper_bgcolor='#0d1117',
    scene=dict(
        xaxis=dict(title='X (km)', backgroundcolor='#0d1117', gridcolor='#333', color='white'),
        yaxis=dict(title='Y (km)', backgroundcolor='#0d1117', gridcolor='#333', color='white'),
        zaxis=dict(title='Z (km)', backgroundcolor='#0d1117', gridcolor='#333', color='white'),
        bgcolor='#0d1117', aspectmode='cube',
        camera=dict(eye=dict(x=1.6, y=1.6, z=0.9))
    ),
    legend=dict(font=dict(color='white', size=8), bgcolor='rgba(0,0,0,0.5)'),
    margin=dict(l=0, r=0, t=40, b=0), height=720
)
fig.write_html('models/orbital_view.html')
fig.show()
print('✅ Interactive 3-D plot saved → models/orbital_view.html')

---
## ⑩ Download All Output Files

---
## ✅ Pipeline Complete!

| Step | Output |
|---|---|
| TLE Download | `data/raw/satellites.csv` |
| Orbit Propagation | `data/processed/trajectories.csv` |
| Feature Engineering | `data/processed/pair_features.csv` |
| Labelled Dataset | `data/processed/labelled_dataset.csv` |
| Trained Model | `models/best_model.joblib` |
| Evaluation Plots | `models/evaluation_plots.png` |
| Monte Carlo Plot | `models/monte_carlo_results.png` |
| 3-D Orbital View | `models/orbital_view.html` |
